# CS 445 — Part 3: Multiclass Classification on Iris

**Goal:** Demonstrate a ScalarFlow multilayer perceptron (MLP) using softmax output and multiclass cross-entropy on the three-class Iris dataset.

This notebook uses the modified `sf_classifiers.py`, `sf_util.py`, and `scalarflow.py` from the Part 3 implementation. Place the notebook in your project directory alongside those files. The Iris data are read from `iris/iris.data` when available; otherwise the equivalent dataset bundled with scikit-learn is used.

**Experiment:** 80% stratified training data, 20% held-out test data, standardization based only on training data, one-hot training targets, and a 3-output MLP. We plot training loss and evaluate the held-out test set with a confusion matrix.

## 1. Import libraries and load Iris data

Each observation has four numeric measurements and one of three class labels: *setosa*, *versicolor*, or *virginica*. The notebook uses the existing ScalarFlow implementation for the neural network, not scikit-learn's MLP.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import sf_classifiers

CLASS_NAMES = ["setosa", "versicolor", "virginica"]
iris_path = Path("iris/iris.data")

if iris_path.is_file():
    rows = np.loadtxt(iris_path, delimiter=",", dtype=str)
    X = rows[:, :4].astype(float)
    label_to_id = {
        "Iris-setosa": 0,
        "Iris-versicolor": 1,
        "Iris-virginica": 2,
    }
    y = np.array([label_to_id[label] for label in rows[:, 4]], dtype=int)
    source = str(iris_path)
else:
    iris = load_iris()
    X, y = iris.data.astype(float), iris.target.astype(int)
    source = "scikit-learn bundled Iris dataset"

print("Data source:", source)
print("Feature matrix shape:", X.shape)
print("Samples per class:", np.bincount(y))

## 2. Split, standardize, and one-hot encode the targets

A stratified 80/20 split keeps class proportions similar in the training and test sets. The scaler is fitted **only** on the training data, avoiding information leakage from the held-out test set.

For three classes, one-hot targets are vectors such as `[1, 0, 0]`, `[0, 1, 0]`, and `[0, 0, 1]`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

NUM_CLASSES = 3
Y_train = np.eye(NUM_CLASSES, dtype=float)[y_train]
Y_test = np.eye(NUM_CLASSES, dtype=float)[y_test]

print("Training samples:", len(X_train))
print("Held-out test samples:", len(X_test))
print("Example class label:", y_train[0])
print("Example one-hot target:", Y_train[0])

## 3. Train the multiclass ScalarFlow MLP

The model uses a ReLU hidden layer and three softmax outputs. Its `train` method performs stochastic gradient descent and returns the loss for each epoch. A relatively small hidden layer keeps training manageable in the scalar-by-scalar implementation.

You can adjust `HIDDEN_SIZES`, `LEARNING_RATE`, or `EPOCHS` if your results suggest the model needs more training. The **test set is not used for weight updates**.

In [ ]:
import contextlib
import io

HIDDEN_SIZES = [10]
ACTIVATION = "relu"
LEARNING_RATE = 0.03
EPOCHS = 100

model = sf_classifiers.MLP(
    input_dim=X_train.shape[1],
    hidden_sizes=HIDDEN_SIZES,
    num_classes=NUM_CLASSES,
    activation=ACTIVATION,
)

# The provided training implementation prints every epoch;
# suppress those lines to keep notebook output readable.
with contextlib.redirect_stdout(io.StringIO()):
    loss_history = model.train(
        X_train, Y_train, learning_rate=LEARNING_RATE, epochs=EPOCHS
    )

assert len(loss_history) == EPOCHS, "Expected one loss value per epoch."
assert np.all(np.isfinite(loss_history)), "Training loss contains NaN or infinity."

print(f"Model: {HIDDEN_SIZES} hidden units, activation={ACTIVATION}")
print(f"Learning rate: {LEARNING_RATE}, epochs: {EPOCHS}")
print(f"First epoch loss: {loss_history[0]:.4f}")
print(f"Final epoch loss: {loss_history[-1]:.4f}")

## 4. Plot training loss

The curve displays the **sum of multiclass cross-entropy losses** across the training examples during each epoch, using the training method provided by the class. Overall downward movement indicates that the model is fitting the training data.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(np.arange(1, len(loss_history) + 1), loss_history, linewidth=2)
ax.set(title="Iris multiclass MLP — training loss",
       xlabel="Epoch", ylabel="Total training cross-entropy loss")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Evaluate on the held-out test set

The MLP returns three predicted probabilities for each Iris sample. We choose the most probable class using `argmax` and compare the predictions to the held-out true labels. Rows in the confusion matrix represent **true** classes; columns represent **predicted** classes.

In [ ]:
test_probs = model.predict(X_test)
assert test_probs.shape == (len(X_test), NUM_CLASSES), (
    "Expected one probability per class for each test sample."
)
assert np.all(np.isfinite(test_probs)), "Predictions contain NaN or infinity."
assert np.allclose(test_probs.sum(axis=1), 1.0, atol=1e-5), (
    "Softmax probabilities should sum to 1."
)

predicted_classes = np.argmax(test_probs, axis=1)
accuracy = accuracy_score(y_test, predicted_classes)
cm = confusion_matrix(y_test, predicted_classes, labels=[0, 1, 2])

print(f"Held-out test accuracy: {accuracy:.3f} ({(predicted_classes == y_test).sum()}/{len(y_test)})")
print("Confusion matrix (true rows, predicted columns):")
print(cm)

display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(6, 5))
display.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.set_title("Iris multiclass MLP — held-out test set")
plt.tight_layout()
plt.show()

## 6. Discussion

After running the notebook, briefly interpret the actual output rather than leaving a generic conclusion. Address:

- Did total training cross-entropy decrease over the epochs?
- What was the model's accuracy on the 30 held-out Iris samples?
- Which classes, if any, were confused? Refer to the off-diagonal confusion-matrix cells.
- Explain that softmax produces a distribution over three classes, multiclass cross-entropy measures prediction error against one-hot targets, and `argmax` selects the predicted label.

**Suggested write-up template (replace brackets after running):**

> The multiclass ScalarFlow MLP was trained on 120 Iris samples and evaluated on 30 held-out samples. With [architecture], [activation] activation, a learning rate of [value], and [number] epochs, training loss changed from [start] to [end]. Test accuracy was [value]. The confusion matrix shows [which classes were mistaken, or no mistakes]. These results demonstrate that the updated ScalarFlow model can classify three classes using softmax output and multiclass cross-entropy.

**Submission check:** Run **Kernel → Restart & Run All**, confirm both figures appear and no exceptions occur, and save the notebook **with its outputs** before submission.